# CoralWatch: preserve the pair within each colony

How can twenty colonies produce forty readings but still only twenty sampled colonies? We use one real opportunistic CoralWatch survey to compare paired charts, distributions and morphology summaries without expanding its sampling claims.

## Objectives

- Validate published paired-score arithmetic
- Show within-colony ranges and between-colony distributions
- Distinguish chart indices from bleaching percentages and reef coverage

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples_v2",
              Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples_v2 in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import importlib
import repo_examples as rx
# Pick up a published helper correction when rerunning in an existing kernel.
rx = importlib.reload(rx)

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("coral"))

In [ ]:
coral = rx.read_csv("coral/coralwatch_random_survey_20.csv")
metadata = rx.read_json("coral/coralwatch_random_survey_20.metadata.json")
light = coral.lightest_colour_score.to_numpy()
dark = coral.darkest_colour_score.to_numpy()
average = coral.average_colour_score.to_numpy()
assert len(coral) == 20 and coral.sample_id.is_unique
assert np.all((light >= 1) & (light <= 6) & (dark >= 1) & (dark <= 6))
assert np.allclose(average, (light+dark)/2)
assert metadata["csv_sha256"] == rx.manifest()["files"]["coral/coralwatch_random_survey_20.csv"]["sha256"]
display(coral[["sample_id", "coral_form", "lightest_colour_score", "darkest_colour_score", "average_colour_score"]])
print(metadata["citation"])
print(metadata["date_interpretation"])

## View 1: one horizontal pair per colony

Each line joins two readings of the same colony, with its published average marked in the middle. Both endpoints remain visible even when their mean matches another colony's. Labels are within-survey sample IDs, not observer identities or verified species. Color here distinguishes measurement roles; it does not reproduce the CoralWatch hue chart.

**Method connection:** compare the catalog's Table (§3), Histogram (§7), Scatterplot (§9) and Tukey box plot (§10). The paired dot/range plot here is an additional technique that preserves within-colony pairing; a bar of the overall mean discards it. The 1–6 chart index is treated descriptively; equal numerical steps need not imply equal physiological changes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(coral))
ax.hlines(y, light, dark, color="#8b9da4", lw=2)
ax.scatter(light, y, color="#d6a848", marker="o", label="Lightest score")
ax.scatter(dark, y, color="#216b82", marker="s", label="Darkest score")
ax.scatter(average, y, color="#8b404f", marker="x", label="Published mean")
ax.set(yticks=y, yticklabels=coral.sample_id, xticks=range(1, 7), xlim=(.7, 6.3))
ax.invert_yaxis()
rx.finish(ax, "20 colonies, paired chart readings", "CoralWatch chart index (1–6)", "Within-survey sample ID")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## View 2: distribution without losing the sample unit

The histogram counts per-colony averages, never forty independent corals. The second panel compares paired span with mean; a wide span can coexist with an intermediate average. This arithmetic inspection validates internal consistency rather than the accuracy of field readings.

In [ ]:
coral["paired_span"] = dark-light
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(average, bins=np.arange(.75, 6.76, .5), color="#54869a", edgecolor="white")
axes[0].set_xticks(range(1, 7))
rx.finish(axes[0], "Published per-colony means", "Mean chart index", "Colonies (n=20)")
axes[1].scatter(average, coral.paired_span, s=65, color="#b2763a", alpha=.65)
for i, row in coral.iterrows():
    axes[1].annotate(str(row.sample_id), (row.average_colour_score, row.paired_span), xytext=(3, 3), textcoords="offset points", fontsize=8)
rx.finish(axes[1], "Pairing reveals within-colony span", "Published mean chart index", "Darkest − lightest (index steps)")
fig.tight_layout()
plt.show()

## View 3: morphology summaries need their denominators

Group sizes differ. The dots retain every colony and the larger diamond marks the group arithmetic mean. A group range is descriptive variation, not a confidence interval, and the few opportunistic samples do not support a reef-wide ranking of coral forms. The leave-one-out calculation shows how one colony can shift this small survey's overall mean without claiming a sampling distribution.

In [ ]:
groups = coral.groupby("coral_form").average_colour_score.agg(["size", "mean", "min", "max"])
display(groups)
fig, ax = plt.subplots(figsize=(10, 4))
for j, (form, group) in enumerate(coral.groupby("coral_form", sort=True)):
    offsets = np.linspace(-.12, .12, len(group))
    ax.scatter(group.average_colour_score, j+offsets, color="#487e95", alpha=.8)
    ax.scatter(group.average_colour_score.mean(), j, marker="D", s=75, color="#b87536", zorder=4)
ax.set(yticks=range(len(groups)), yticklabels=[f"{name} (n={int(row['size'])})" for name, row in groups.iterrows()], xlim=(1, 6))
rx.finish(ax, "Observed colonies and descriptive group means", "Per-colony mean chart index", "Published morphology category")
fig.tight_layout()
plt.show()
leave_one_out = (average.sum()-average)/(len(average)-1)
print(f"Survey mean = {average.mean():.3f}; leave-one-colony-out means = {leave_one_out.min():.3f} to {leave_one_out.max():.3f}")

## Exercise

Find two colonies with equal means but different paired spans. Explain what a single bar of the mean hides. Replace group means with medians and retain n beside each group. State which conclusions remain valid for this submitted survey and which would require a designed, repeated reef sampling program.

## Limitations and reflection

One opportunistic 20-colony survey is not representative of all corals at Abu Sawatyr, Egypt or any reef. Chart indices are not percent bleaching, coral coverage, temperature, disease or Degree Heating Weeks. Published time fields are preserved without resolving their local time zone. Location accuracy units are unverified; no coordinates, people or photographs are bundled. Arithmetic checks do not independently verify observations.

## Citations and repository connections

CoralWatch (2026), [BioCollect survey bd39e449-038f-4d45-ab6d-b56922a6f532](https://biocollect.ala.org.au/coralwatch/bioActivity/index/bd39e449-038f-4d45-ab6d-b56922a6f532), accessed 2026-10-09; reduced extract under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes: numeric scores extracted from chart codes; personal fields, exact coordinates and photographs omitted. [CoralWatch methods](https://coralwatch.org/faqs/). Retained from [Climate-and-Health 248f9860d189](https://github.com/jltobias/JupyterLite-Climate-and-Health/tree/248f9860d189502013b36b3c4271777edafc4f7f/content/data); full source attribution and transformation metadata are packaged.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.